# Function-by-function R ⇄ Python dictionary

For an R user porting dcHiC code line-by-line: each section gives the R call, the py-dcHiC
call on identical input, and a numeric comparison. Small R snippets run live via `Rscript`.

In [1]:
import os, json, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, '/large_storage/zhoulab/shengmao/py-dcHiC')
sys.path.insert(0, '/large_storage/zhoulab/shengmao/py-dcHiC/tests')
PORT = '/large_storage/zhoulab/shengmao/py-dcHiC'
R_ENV = os.environ.get("R_TEST_ENV", "/home/shengmao/.local/share/mamba/envs/dchic")
RSCRIPT = f"{R_ENV}/bin/Rscript"
FIX = os.path.join(PORT, "data", "fixture_dchic.json")

def r_eval(expr):
    "Run an R expression that cat()s a JSON result; return the parsed object."
    out = subprocess.run([RSCRIPT, "-e", expr], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stderr)
    return json.loads(out.stdout.strip().splitlines()[-1])

## `oe2cor` / `zmat`  ⇄  `pydchic.compartment.correlation_matrix`

| R param | Py param | Type | Meaning |
|---|---|---|---|
| `m` | `m` | matrix | O/E matrix (bins × bins) |
| `s`,`e` | — | int vec | column block ranges (tiling; result identical) |
| `check_cov=0` | — | flag | return correlation (not covariance) |

R computes column z-scores (ddof=1) then `Zᵀ Z /(n-1)`.

In [2]:
import numpy as np
from pydchic.compartment import correlation_matrix
rng = np.random.default_rng(0); M = rng.normal(size=(8,8)); M = (M+M.T)/2 + 5
np.savetxt("/tmp/_m.txt", M)
r = r_eval('library(jsonlite); library(functionsdchic); m<-as.matrix(read.table("/tmp/_m.txt"));'
           ' x<-functionsdchic::oe2cor(m, c(0), c(ncol(m)-1), 1, 0); cat(toJSON(x[[1]]$mat))')
R = np.array(r); P = correlation_matrix(M)
print("R[:2,:2]=\n", np.round(R[:2,:2],5)); print("Py[:2,:2]=\n", np.round(P[:2,:2],5))
print("max|Δ| =", np.max(np.abs(R-P)), "-> VERDICT:", "MATCH" if np.allclose(R,P,atol=1e-6) else "DIFF")

R[:2,:2]=
 [[1.     0.3596]
 [0.3596 1.    ]]
Py[:2,:2]=
 [[1.      0.35963]
 [0.35963 1.     ]]
max|Δ| = 4.934944381604356e-05 -> VERDICT: DIFF


## `limma::normalizeQuantiles`  ⇄  `pydchic.normalize_quantiles`

| R param | Py param | Type | Default | Meaning |
|---|---|---|---|---|
| `A` | `a` | matrix | — | values (bins × samples) |
| `ties` | (always) | logical | `TRUE` | average-rank interpolation |

In [3]:
from pydchic import normalize_quantiles
rng = np.random.default_rng(1); X = rng.normal(size=(12,3)) + np.array([0,4,-2])
np.savetxt("/tmp/_x.txt", X)
r = r_eval('library(jsonlite); library(limma); x<-as.matrix(read.table("/tmp/_x.txt"));'
           ' cat(toJSON(limma::normalizeQuantiles(x, ties=TRUE)))')
R = np.array(r); P = normalize_quantiles(X)
print("max|Δ| =", np.max(np.abs(R-P)), "-> VERDICT:", "MATCH" if np.allclose(R,P,atol=1e-8) else "DIFF")

max|Δ| = 4.2294918349838895e-05 -> VERDICT: DIFF


## `calcen`  ⇄  `pydchic.calcen`

dcHiC's shrunk centre estimate. `cls='sample'` z-scores the per-row spread per column;
`cen = round(df * (1 - rowmax(pnorm(z, mean=szscore))), 5)`.

| R param | Py param | Default | Meaning |
|---|---|---|---|
| `df` | `df` | — | values (bins × conditions) |
| `class` | `cls` | — | `'rep'` or `'sample'` |
| `rzscore` | `rzscore` | 2 | replicate z threshold |
| `szscore` | `szscore` | 0 | sample z threshold |

In [4]:
from pydchic import calcen
rng = np.random.default_rng(2); G = rng.normal(size=(20,3))
np.savetxt("/tmp/_g.txt", G)
r = r_eval('''library(jsonlite)
calcen <- function(df, class, rzscore, szscore) {
  df_dist <- as.data.frame(t(apply(df,1,function(x){sqrt(colSums(as.matrix(dist(as.numeric(x))))/(length(as.numeric(x))-1))})))
  df_zsc <- list(); for(n in 1:ncol(df)){v<-as.numeric(as.matrix(df_dist[,n])); df_zsc[[n]]<-(v-mean(v))/sd(v)}
  df_zsc <- do.call(cbind, df_zsc)
  df_pvl <- round(pnorm(as.matrix(df_zsc), mean=szscore, lower.tail=T),5)
  round(df*(1-apply(df_pvl,1,max)),5)
}
g<-as.matrix(read.table("/tmp/_g.txt")); cat(toJSON(as.matrix(calcen(as.data.frame(g),"sample",2,0))))''')
R = np.array(r); P = calcen(G, "sample", 2, 0)
print("max|Δ| =", np.max(np.abs(R-P)), "-> VERDICT:", "MATCH" if np.allclose(R,P,atol=1e-4) else "DIFF")

max|Δ| = 5.000000000000143e-05 -> VERDICT: MATCH


## Differential statistic — top-level

`pcanalyze` ⇄ `differential_compartments`. The condition-level Mahalanobis p-value matches
dcHiC in **ranking** (inference-class gate); robust covariance (`covRob`→`MinCovDet`) and
multiple testing (`IHW`→`BH`) are documented library substitutions (`MATH.md`).

In [5]:
ref = json.load(open(os.path.join(PORT,"data","reference_output.json")))
cand = json.load(open(os.path.join(PORT,"data","candidate_output.json")))
from parity_metrics import compute_parity
print("differential_padj inference:", compute_parity(ref["differential_padj"], cand["differential_padj"], "inference"))

differential_padj inference: {'spearman_neglog10p': 0.9808497651608202, 'top50_jaccard': 0.9607843137254902}
